# 01 · Preprocess GRID (CPU)
Downloads GRID from the official Sheffield site into `/tmp`, extracts 96×96 mouth ROIs + lip landmarks for **one shard**, and writes them to `/kaggle/working/grid_rois_shard{SHARD}`.

Run this notebook once per shard (`SHARD = 0`, then `1`) with **Save Version → Save & Run All**. Then create a Kaggle Dataset from each version's output (e.g. `grid-rois-shard0`, `grid-rois-shard1`).

Settings: Accelerator **None**, Internet **On**.

In [ ]:
SHARD, NUM_SHARDS = 0, 2      # change to 1 for the second run

In [ ]:
# --- setup: clone the project code (private repo: add a Kaggle secret GITHUB_TOKEN) ---
import os, subprocess
url = "https://github.com/durgaanuhya02/MoVi-Swin-Motion--and-Viseme-Aware-Space-Time-Swin-Transformer-for-Robust-and-Efficient-Lip-Reading.git"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = url.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
!git log --oneline -1

In [ ]:
!pip install -q mediapipe cmudict

In [ ]:
!df -h /tmp /kaggle/working
!nproc

In [ ]:
!GRID_DEST=/tmp/GRID GRID_TMP=/tmp/grid_zips bash download_grid.sh 2>&1 | grep -E 'done|FAILED|FINISHED'

In [ ]:
!python -m src.preprocess.build_grid_manifest --grid-root /tmp/GRID --out /kaggle/working/grid_manifest.jsonl

In [ ]:
OUT = f'/kaggle/working/grid_rois_shard{SHARD}'
!python -m src.preprocess.extract_rois --manifest /kaggle/working/grid_manifest.jsonl --out {OUT} --workers 4 --detect-every 2 --shard {SHARD}/{NUM_SHARDS} 2>&1 | grep -E 'clips|eta' | tail -20

In [ ]:
!wc -l {OUT}/index.jsonl {OUT}/errors.jsonl; du -sh {OUT}
!rm -rf /kaggle/working/repo   # keep only the data in the saved output